# 3.2 — Data Preparation

**Goal:** Telco Customer Churn dataset-কে Machine Learning-এর জন্য প্রস্তুত করা।

এই অংশে data download/read করব, column ও text value uniform করব, `totalcharges` ঠিক করব এবং target `churn`-কে numeric বানাব।

## 1. প্রয়োজনীয় libraries import করা

**কেন?** `pandas` দিয়ে dataset handle করব, `numpy` numerical কাজের জন্য এবং `matplotlib` পরের analysis/visualization-এর কাজে লাগবে।

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Dataset download করা



In [2]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'

# !wget $data -O data-week-3.csv

## 3. CSV পড়া ও data দেখা


In [3]:
df = pd.read_csv('data-week-3.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


`df.head()`-এ অনেক column থাকলে মাঝের column-গুলো `...` দিয়ে লুকানো থাকতে পারে। Transpose করলে row ও column-এর অবস্থান বদলে যায়, তাই সব column-এর value একসঙ্গে দেখা সহজ হয়।

In [4]:
df.shape

(7043, 21)

In [5]:
df.head().T

,0,1,2,3,4
customerID,7590-VHVEG,5575-GNVDE,3668-QPYBK,7795-CFOCW,9237-HQITU
gender,Female,Male,Male,Male,Female
SeniorCitizen,0,0,0,0,0
Partner,Yes,No,No,No,No
Dependents,No,No,No,No,No
tenure,1,34,2,45,2
PhoneService,No,Yes,Yes,No,Yes
MultipleLines,No phone service,No,No,No phone service,No
InternetService,DSL,DSL,DSL,DSL,Fiber optic
OnlineSecurity,No,Yes,Yes,Yes,No


## 4. Column name ও text value uniform করা


In [6]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

## 5. Data type পরীক্ষা করা

**কেন?** কোনো column-এর value দেখতে number-এর মতো হলেও pandas সেটিকে string হিসেবে পড়েছে কি না, তা বুঝতে চাই। বিশেষ করে `totalcharges` numeric হওয়া উচিত।

In [7]:
df.dtypes

customerid           object
gender               object
seniorcitizen         int64
partner              object
dependents           object
tenure                int64
phoneservice         object
multiplelines        object
internetservice      object
onlinesecurity       object
onlinebackup         object
deviceprotection     object
techsupport          object
streamingtv          object
streamingmovies      object
contract             object
paperlessbilling     object
paymentmethod        object
monthlycharges      float64
totalcharges         object
churn                object
dtype: object

## 6. `totalcharges`-কে numeric-এ রূপান্তর

Original data-তে কিছু missing value ফাঁকা space দিয়ে লেখা ছিল। আগের step-এ space `_` হয়ে যাওয়ায় `totalcharges`-এ কিছু value `_` হয়েছে; pandas এটিকে number হিসেবে convert করতে পারে না।

In [8]:
df['totalcharges']
# shob to number ei dekha jacche tai na? then numeric e convert kore dekhi tahole ei column ke object boltese ken

0         29.85
1        1889.5
2        108.15
3       1840.75
4        151.65
         ...   
7038     1990.5
7039     7362.9
7040     346.45
7041      306.6
7042     6844.5
Name: totalcharges, Length: 7043, dtype: object

In [9]:
# pd.to_numeric(df['totalcharges'])

In [10]:
# so uporer error dekhay: ValueError: Unable to parse string "_" at position 488
# mane 488 index e "_" ache, tai numeric e convert korte parlo na. so ei row ta dekhbo. maybe aaro emon ase. but error er first index tai show kore.

df['totalcharges'].iloc[488]

'_'

In [11]:
pd.to_numeric(df['totalcharges'], errors='coerce')

# now error gone.

0         29.85
1       1889.50
2        108.15
3       1840.75
4        151.65
         ...   
7038    1990.50
7039    7362.90
7040     346.45
7041     306.60
7042    6844.50
Name: totalcharges, Length: 7043, dtype: float64

`errors='coerce'` invalid value-কে `NaN` বানায়, ফলে conversion error দিয়ে থেমে যায় না। এরপর কতগুলো missing value হয়েছে, তা গুনি।

In [12]:
tc = pd.to_numeric(df['totalcharges'], errors='coerce')

In [13]:
tc.isnull().sum()

np.int64(11)

## 7. Missing row দেখা ও ঠিক করা

**কেন?** কোন customer-দের `totalcharges` missing, তা পরীক্ষা করে তারপর উপযুক্ত value বসাব।

In [14]:
# 11 ta row te null value ase. oi 11 ta row dekhbo.

df[tc.isnull()]

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
488,4472-lvygi,female,0,yes,yes,0,no,no_phone_service,dsl,yes,...,yes,yes,yes,no,two_year,yes,bank_transfer_(automatic),52.55,_,no
753,3115-czmzd,male,0,no,yes,0,yes,no,no,no_internet_service,...,no_internet_service,no_internet_service,no_internet_service,no_internet_service,two_year,no,mailed_check,20.25,_,no
936,5709-lvoeq,female,0,yes,yes,0,yes,no,dsl,yes,...,yes,no,yes,yes,two_year,no,mailed_check,80.85,_,no
1082,4367-nuyao,male,0,yes,yes,0,yes,yes,no,no_internet_service,...,no_internet_service,no_internet_service,no_internet_service,no_internet_service,two_year,no,mailed_check,25.75,_,no
1340,1371-dwpaz,female,0,yes,yes,0,no,no_phone_service,dsl,yes,...,yes,yes,yes,no,two_year,no,credit_card_(automatic),56.05,_,no
3331,7644-omvmy,male,0,yes,yes,0,yes,no,no,no_internet_service,...,no_internet_service,no_internet_service,no_internet_service,no_internet_service,two_year,no,mailed_check,19.85,_,no
3826,3213-vvolg,male,0,yes,yes,0,yes,yes,no,no_internet_service,...,no_internet_service,no_internet_service,no_internet_service,no_internet_service,two_year,no,mailed_check,25.35,_,no
4380,2520-sgtta,female,0,yes,yes,0,yes,no,no,no_internet_service,...,no_internet_service,no_internet_service,no_internet_service,no_internet_service,two_year,no,mailed_check,20.00,_,no
5218,2923-arzlg,male,0,yes,yes,0,yes,no,no,no_internet_service,...,no_internet_service,no_internet_service,no_internet_service,no_internet_service,one_year,yes,mailed_check,19.70,_,no
6670,4075-wkniu,female,0,yes,yes,0,yes,yes,dsl,no,...,yes,yes,yes,no,two_year,no,mailed_check,73.35,_,no


In [15]:
df[tc.isnull()][['customerid', 'totalcharges']]

,customerid,totalcharges
488,4472-lvygi,_
753,3115-czmzd,_
936,5709-lvoeq,_
1082,4367-nuyao,_
1340,1371-dwpaz,_
3331,7644-omvmy,_
3826,3213-vvolg,_
4380,2520-sgtta,_
5218,2923-arzlg,_
6670,4075-wkniu,_


এই dataset-এ এমন ১১টি row আছে। এগুলো সাধারণত নতুন customer, যাদের total charge রেকর্ড হয়নি। তাই এখানে missing `totalcharges`-কে `0` দিয়ে পূরণ করছি। অন্য dataset-এ সবসময় `0` উপযুক্ত নাও হতে পারে।

In [16]:
df['totalcharges'] = pd.to_numeric(df['totalcharges'], errors='coerce').fillna(0)

## 8. Target variable `churn` numeric করা

**কেন?** `churn`-এ `yes`/`no` string আছে। Classification model-এর জন্য এটিকে `1`/`0` করা দরকার।

- `yes` → `1`: customer churn করেছে।
- `no` → `0`: customer churn করেনি।

In [17]:
df.churn.head()

0     no
1     no
2    yes
3     no
4    yes
Name: churn, dtype: object

In [18]:
df['churn'] = (df['churn'] == 'yes').astype(int)
df['churn'].head()

0    0
1    0
2    1
3    0
4    1
Name: churn, dtype: int64

In [19]:
df.dtypes

customerid           object
gender               object
seniorcitizen         int64
partner              object
dependents           object
tenure                int64
phoneservice         object
multiplelines        object
internetservice      object
onlinesecurity       object
onlinebackup         object
deviceprotection     object
techsupport          object
streamingtv          object
streamingmovies      object
contract             object
paperlessbilling     object
paymentmethod        object
monthlycharges      float64
totalcharges        float64
churn                 int64
dtype: object

# 3.3 — Setting Up the Validation Framework

## 1. `train_test_split` import করা

**কেন?** Scikit-Learn-এর `train_test_split()` দিয়ে dataset-কে randomভাবে দুই ভাগে ভাগ করা যায়। একবারে তিন ভাগ হয় না, তাই দুই ধাপে split করব।

In [20]:
from sklearn.model_selection import train_test_split

## 2. Dataset-কে 60% / 20% / 20% ভাগ করা

প্রথমে 20% `test` set আলাদা করি। বাকি 80% থাকবে `df_full_train`-এ।

- `test_size=0.2` → মোট data-এর 20% test set-এ যায়।
- `random_state=1` → একই seed থাকলে প্রতিবার একই split পাওয়া যায়।

In [21]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)

এখন বাকি 80% থেকে `train` ও `validation` set বানাই।

**কেন `test_size=0.25`?** এই ধাপের input হলো মূল dataset-এর 80%। তার 25% হলো মূল dataset-এর 20% — তাই validation set মোট data-এর 20% হবে।

In [22]:
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

## 3. Split-এর size পরীক্ষা করা

Expected output: `(4225, 1409, 1409)`

- `df_train`: 4,225 rows (প্রায় 60%)
- `df_val`: 1,409 rows (প্রায় 20%)
- `df_test`: 1,409 rows (প্রায় 20%)

In [23]:
len(df_train), len(df_val), len(df_test)

(4225, 1409, 1409)

## 4. Index reset করা

`train_test_split()` row shuffle করে, তাই পুরোনো index এলোমেলো থাকতে পারে। eta model e provab fele na. tai eta optional

In [24]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

df_full_train = df_full_train.reset_index(drop=True)

## 5. Target `churn` আলাদা করা

In [25]:
y_train = df_train.churn.to_numpy()
y_val = df_val.churn.to_numpy()
y_test = df_test.churn.to_numpy()

এরপর তিনটি dataframe থেকে `churn` column মুছে ফেলি।

**কেন?** `churn`-কে input feature হিসেবে রেখে দিলে model উত্তরটি আগেই পেয়ে যেতে পারে — একে **target leakage** বলে।

In [26]:
del df_train['churn']
del df_val['churn']
del df_test['churn']

**মনে রাখবে:** `df_full_train`-এ `churn` রাখা হয়েছে, কারণ পরের EDA ধাপে target-এর distribution ও churn rate বিশ্লেষণ করতে হবে। EDA শুরু করার সময় এর index-ও reset করা হবে।

# 3.4 — EDA (Exploratory Data Analysis)

এখানে `df_full_train` ব্যবহার করছি (train + validation data)। `df_test` শেষ evaluation-এর আগে আলাদা রাখব।

## 1. Missing values check করা

In [27]:
df_full_train.isnull().sum()

# kono column ei r missing value nai. karon amra agei resolve korsi.

customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        0
churn               0
dtype: int64

## 3. Churn rate বোঝা

`churn` হলো target column:

- `0` = customer থেকে গেছে
- `1` = customer চলে গেছে (churn করেছে)



### 3.1 প্রতিটি value-এর proportion দেখা

`normalize=True` দিলে `value_counts()` শুধু count না দিয়ে total rows-এর অনুপাতে fraction দেখায়।

উদাহরণস্বরূপ result:

- `0 → 0.730` অর্থ প্রায় **73% customer থেকে গেছে**।
- `1 → 0.270` অর্থ প্রায় **27% customer churn করেছে**।

এই `1`-এর proportion-টাই **churn rate**। Percentage-এ পেতে `0.270 × 100 ≈ 27%`।

In [28]:
df_full_train['churn'].value_counts(normalize=True)

churn
0    0.730032
1    0.269968
Name: proportion, dtype: float64

### 3.2 `churn.mean()` কেন churn rate দেয়?

`churn` column-এ শুধু `0` ও `1` আছে। Mean = সব value-এর যোগফল ÷ মোট value-এর সংখ্যা। `0` যোগফলে কিছু যোগ করে না; প্রতিটি `1` এক যোগ করে। তাই:

$$
\text{mean} = \frac{\text{number of 1s}}{\text{total rows}} = \text{churn rate}
$$

**সহজ উদাহরণ:** `[0, 0, 1, 0, 1]`—৫ জনের মধ্যে ২ জন churn করেছে। Mean = `(0+0+1+0+1) / 5 = 2/5 = 0.4`, অর্থাৎ churn rate **40%**।

In [29]:
df_full_train['churn'].mean()

np.float64(0.26996805111821087)

এই dataset-এ result প্রায় `0.270`, অর্থাৎ churn rate প্রায় **27%**। তাই binary (0/1) column-এর mean দিয়ে 1-এর proportion বের করা যায়।

## 4. Numerical ও categorical columns আলাদা করা

In [30]:
df_full_train.columns

Index(['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents',
       'tenure', 'phoneservice', 'multiplelines', 'internetservice',
       'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport',
       'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling',
       'paymentmethod', 'monthlycharges', 'totalcharges', 'churn'],
      dtype='object')

### Numerical columns

- `tenure`: customer কত মাস ধরে আছে
- `monthlycharges`: প্রতি মাসের charge
- `totalcharges`: মোট charge

`seniorcitizen` দেখতে `0/1` হলেও এটি আসলে category (senior citizen কি না)। `churn` হলো target, আর `customerid` শুধু identifier—সাধারণ feature হিসেবে ব্যবহার করব না।

In [31]:
numerical = ['tenure', 'monthlycharges', 'totalcharges']

### Categorical columns

In [32]:
categorical = [
    'gender',
    'seniorcitizen',
    'partner',
    'dependents',
    'phoneservice',
    'multiplelines',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'contract',
    'paperlessbilling',
    'paymentmethod',
]

# 3.5 — Feature Importance: Churn Rate & Risk Ratio

**Goal:** কোন categorical feature-এর কোন group-এ customer churn বেশি বা কম, তা global churn rate-এর সঙ্গে তুলনা করা।

এই notebook-এ 3.3–3.4 থেকে পাওয়া `df_full_train` এবং `categorical` list ব্যবহার করা হয়েছে। `churn` আগে থেকেই `0` (থেকে গেছে) ও `1` (চলে গেছে) হিসেবে encoded থাকতে হবে।

## 1. Global churn rate বের করা

```python
global_churn = df_full_train.churn.mean()
```

**কেন?** এটি পুরো `df_full_train` dataset-এর average churn rate; প্রতিটি group-এর সঙ্গে তুলনা করার reference point। `churn`-এ 1 মানে churn করেছে এবং 0 মানে করেনি, তাই mean-ই churn করা customer-দের proportion।

In [33]:
global_churn = df_full_train.churn.mean()
global_churn

np.float64(0.26996805111821087)

প্রত্যাশিত ফল প্রায় **0.27 = 27%**। অর্থাৎ পুরো dataset-এ প্রতি 100 জন customer-এর মধ্যে প্রায় 27 জন churn করেছে।

## 2. একটি feature দিয়ে শুরু: `partner`

আগে loop ব্যবহার না করে শুধু `partner` column দেখি। এখানে দুইটি group: `yes` এবং `no`।

### 2.1 প্রতিটি group-এ কতজন customer আছে?

In [34]:
df_full_train.partner.value_counts()

partner
no     2932
yes    2702
Name: count, dtype: int64

এতে প্রতিটি category-র customer count পাওয়া যায়। Count জানা থাকলে rate কতজন customer-এর ওপর ভিত্তি করে এসেছে, সেটাও বোঝা যায়।

### 2.2 Group অনুযায়ী churn rate বের করা

**কেন?** প্রথমে `partner == 'yes'` দিয়ে সেই group-এর rows filter করি; তারপর তাদের `churn.mean()` হিসাব করি। একইভাবে `partner == 'no'` group-এর rate বের করি।

In [35]:
churn_partner = df_full_train[df_full_train.partner == 'yes'].churn.mean()
churn_no_partner = df_full_train[df_full_train.partner == 'no'].churn.mean()

churn_partner, churn_no_partner

(np.float64(0.20503330866025166), np.float64(0.3298090040927694))

ফল প্রায়:

- `partner = yes` → **20.5%** churn
- `partner = no` → **33.0%** churn
- Global churn rate → **27.0%**

অর্থাৎ partner নেই এমন customer-দের churn rate global average-এর চেয়ে বেশি; partner আছে এমন customer-দের rate কম।

### 2.3 Difference দিয়ে global rate-এর সঙ্গে তুলনা

এখানে একটাই formula ব্যবহার করব:

$$\text{diff} = \text{group churn rate} - \text{global churn rate}$$

এই convention সহজে interpret করা যায়:

- `diff > 0` → group-এর churn rate global average-এর **চেয়ে বেশি**।
- `diff < 0` → group-এর churn rate global average-এর **চেয়ে কম**।
- `diff = 0` → group ও global churn rate সমান।

Difference-এর absolute value যত বড়, group-এর rate global average থেকে তত বেশি দূরে।

In [36]:
diff_partner = churn_partner - global_churn
diff_no_partner = churn_no_partner - global_churn

diff_partner, diff_no_partner

(np.float64(-0.06493474245795922), np.float64(0.05984095297455855))

- `partner=yes`: `0.2050 - 0.2700 ≈ -0.0649` → global average-এর চেয়ে প্রায় **6.5 percentage points কম** churn।
- `partner=no`: `0.3298 - 0.2700 ≈ +0.0598` → global average-এর চেয়ে প্রায় **6.0 percentage points বেশি** churn।

এখানে `diff`-এর sign সরাসরি direction জানায়: **positive = বেশি churn, negative = কম churn**।

### 2.4 Risk ratio দিয়ে তুলনা

Difference absolute gap দেখায়। Risk ratio দেখায় group-এর churn rate global rate-এর তুলনায় কত গুণ।

$$
\text{risk ratio} = \frac{\text{group churn rate}}{\text{global churn rate}}
$$

মনে রাখো:

- `risk = 1` → global average-এর সমান
- `risk > 1` → global average-এর চেয়ে বেশি churn
- `risk < 1` → global average-এর চেয়ে কম churn

In [37]:
risk_no_partner = churn_no_partner / global_churn
risk_partner = churn_partner / global_churn

risk_no_partner, risk_partner

(np.float64(1.2216593879412643), np.float64(0.7594724924338315))

ফল প্রায়:

- `partner=no` → **1.22**: global average-এর তুলনায় প্রায় **22% বেশি** churn।
- `partner=yes` → **0.76**: global average-এর তুলনায় প্রায় **24% কম** churn।

যেমন `1.22` মানে 122% churn rate নয়; এটি global rate-এর **1.22 গুণ** বোঝায়।

## 3. SQL `GROUP BY` দিয়ে একই calculation

একটি feature-এর category অনুযায়ী group করে churn rate, difference ও risk ratio বের করার একই idea SQL দিয়েও লেখা যায়।

### SQL code

```sql
SELECT
    gender,
    AVG(churn),
    AVG(churn) - global_churn AS diff,
    AVG(churn) / global_churn AS risk
FROM
    data
GROUP BY
    gender;
```

**কেন প্রতিটি line লিখেছি?**

- `SELECT gender` — gender অনুযায়ী result চাই।
- `AVG(churn)` — প্রতিটি gender group-এর churn rate; কারণ `churn` শুধু `0` বা `1`, তাই average-ই churn করা customer-এর proportion।
- `AVG(churn) - global_churn AS diff` — **group rate − global rate**। Positive হলে group-এর churn বেশি, negative হলে কম।
- `AVG(churn) / global_churn AS risk` — group rate global rate-এর কত গুণ, তা বের করে।
- `FROM data` — source table নির্ধারণ করে।
- `GROUP BY gender` — একই gender-এর row এক group-এ এনে আলাদা হিসাব করে।

**নোট:** এই উদাহরণে `global_churn`-কে আগে থেকে হিসাব করা scalar value ধরা হয়েছে। বাস্তব SQL query-তে সেটি database অনুযায়ী variable, CTE বা subquery দিয়ে define করতে হবে।

## 4. Pandas `groupby()` দিয়ে `partner`-এর table

SQL-এর মতোই Pandas-এ `groupby()` category অনুযায়ী group তৈরি করে। প্রথমে শুধু `partner` দিয়ে হিসাব করি, তারপর `diff` ও `risk` যোগ করি।

In [38]:
df_group = df_full_train.groupby('partner').churn.agg(['mean', 'count'])
df_group

,mean,count
partner,,
no,0.329809,2932
yes,0.205033,2702


- `groupby('partner')` → `yes` ও `no` group তৈরি করে।
- `mean` → প্রতিটি group-এর churn rate।
- `count` → প্রতিটি group-এ customer সংখ্যা।

এখন table-এ difference ও risk ratio যোগ করব। উপরের manual calculation-এর মতোই এখানে `diff = group mean - global churn` ব্যবহার করব, তাই positive/negative sign-এর অর্থ সব জায়গায় একই থাকবে।

In [39]:
df_group['diff'] = df_group['mean'] - global_churn
df_group['risk'] = df_group['mean'] / global_churn
df_group

,mean,count,diff,risk
partner,,,,
no,0.329809,2932,0.059841,1.221659
yes,0.205033,2702,-0.064935,0.759472


এই table-এ একই interpretation প্রযোজ্য:

- `diff > 0` → group global average-এর চেয়ে **বেশি** churn করে।
- `diff < 0` → group global average-এর চেয়ে **কম** churn করে।
- `risk > 1` → group-এর churn rate global average-এর চেয়ে বেশি।
- `risk < 1` → group-এর churn rate global average-এর চেয়ে কম।

ফলে `partner=no`-এর `diff` প্রায় `+0.0598`, আর `partner=yes`-এর `diff` প্রায় `-0.0649`।

## 5. Loop দিয়ে সব categorical feature বিশ্লেষণ

একটি column-এর `groupby()` table বোঝার পর একই হিসাব সব categorical column-এর জন্য চালাই। এখানে `diff` formula আগের মতোই `group mean - global churn` থাকবে।

In [40]:
from IPython.display import display

for c in categorical:
    print(c)
    df_group = df_full_train.groupby(c).churn.agg(['mean', 'count'])
    df_group['diff'] = df_group['mean'] - global_churn
    df_group['risk'] = df_group['mean'] / global_churn
    display(df_group)
    print()

gender


,mean,count,diff,risk
gender,,,,
female,0.276824,2796,0.006856,1.025396
male,0.263214,2838,-0.006755,0.974980



seniorcitizen


,mean,count,diff,risk
seniorcitizen,,,,
0,0.242270,4722,-0.027698,0.897403
1,0.413377,912,0.143409,1.531208



partner


...

,mean,count,diff,risk
contract,,,,
month-to-month,0.431701,3104,0.161733,1.599082
one_year,0.120573,1186,-0.149395,0.446621
two_year,0.028274,1344,-0.241694,0.104730



paperlessbilling


,mean,count,diff,risk
paperlessbilling,,,,
no,0.172071,2313,-0.097897,0.637375
yes,0.338151,3321,0.068183,1.252560



paymentmethod


,mean,count,diff,risk
paymentmethod,,,,
bank_transfer_(automatic),0.168171,1219,-0.101797,0.622928
credit_card_(automatic),0.164339,1217,-0.105630,0.608733
electronic_check,0.455890,1893,0.185922,1.688682
mailed_check,0.193870,1305,-0.076098,0.718121




`display()` ব্যবহার করা হয়েছে, কারণ loop-এর ভেতরে DataFrame শুধু লিখলে প্রতিটি table সুন্দরভাবে display নাও হতে পারে।

# 3.6 — Feature Importance: Mutual Information

**Goal:** কোন categorical feature থেকে `churn` সম্পর্কে কতটা তথ্য পাওয়া যায়, তা একটি score দিয়ে মাপা এবং সব feature-কে বেশি থেকে কম গুরুত্বপূর্ণ ক্রমে সাজানো।

**Prerequisite:** আগের notebook-এর `df_full_train` এবং `categorical` list তৈরি করা থাকতে হবে।

## Mutual Information (MI)

**Mutual Information (MI)** measures how much information one variable provides about another variable.

আমাদের project-এ আমরা জানতে চাই: কোনো customer-এর একটি feature জানলে তার `churn` সম্পর্কে কতটা ধারণা পাওয়া যায়?

### 1. Understanding Entropy

**Entropy** means uncertainty.

- `H(Y)` = `churn` সম্পর্কে uncertainty, কোনো feature না জেনে।
- `H(Y | X)` = feature `X` জানার পরেও `churn` সম্পর্কে যতটুকু uncertainty থাকে।
- `I(X; Y)` = feature `X` জানার কারণে `churn` সম্পর্কে যতটুকু uncertainty কমে।

এখানে:
- `X` = কোনো feature, যেমন `contract`
- `Y` = target variable, অর্থাৎ `churn`

### 2. Mutual Information Formula

$$
I(X;Y) = H(Y) - H(Y|X)
$$

সহজভাবে:

**Mutual Information = আগের uncertainty − feature জানার পরের uncertainty**

### 3. সহজ উদাহরণ

ধরো, কোনো customer-এর contract type জানার আগে আমরা শুধু জানি যে dataset-এ overall churn rate প্রায় 27%।

এখন contract type জানার পর দেখা গেল:
- `month-to-month` → churn rate প্রায় 43%
- `one_year` → churn rate প্রায় 12%
- `two_year` → churn rate প্রায় 3%

Contract type জানার পর customer-এর churn সম্পর্কে আমাদের ধারণা অনেক বেশি নির্দিষ্ট হলো। তাই `contract`-এর Mutual Information বেশি।

অন্যদিকে, নারী ও পুরুষ customer-এর churn rate প্রায় একই। Gender জানলে churn সম্পর্কে ধারণা খুব সামান্য বদলায়। তাই `gender`-এর Mutual Information প্রায় শূন্য।

### 4. MI Score কীভাবে বুঝবে?

- **High MI** → feature জানলে target সম্পর্কে অনেক information পাওয়া যায়।
- **Low MI** → feature জানলে target সম্পর্কে খুব কম information পাওয়া যায়।
- **MI = 0** → feature ও target-এর মধ্যে কোনো statistical information/association নেই।

**মনে রাখবে:** MI score বেশি মানেই feature সরাসরি churn ঘটায়—এমন নয়। এটি feature ও target-এর মধ্যে information-এর পরিমাণ বোঝায়।

## 2. Risk Ratio থেকে পার্থক্য

আগের অংশে `risk ratio` দিয়ে একটি feature-এর category-গুলোর churn rate global churn rate-এর সঙ্গে তুলনা করেছি। Mutual Information আলাদা: এটি **প্রতিটি categorical feature-এর জন্য একটি score** দেয়, তাই feature-গুলোকে একই scale-এ rank করা যায়।

উদাহরণ হিসেবে, এই dataset-এ `contract`-এর MI প্রায় `0.0983`, কিন্তু `gender`-এর MI প্রায় `0.0001`। তাই `contract` churn সম্পর্কে অনেক বেশি তথ্য দেয়; `gender` খুব কম দেয়।

## 3. Scikit-learn function import

In [41]:
from sklearn.metrics import mutual_info_score

`mutual_info_score(x, y)` দুইটি categorical variable-এর mutual information score বের করে। আমাদের ক্ষেত্রে একটি variable হলো feature এবং অন্যটি হলো `df_full_train.churn`। Argument-এর order বদলালেও score একই থাকে, কারণ MI symmetric।

## 4. সব categorical feature-এর MI বের করা

একেকটি column-এর জন্য আলাদা করে code না লিখে, একটি function তৈরি করে `apply()` দিয়ে সব categorical column-এ চালাব।

In [42]:
def mutual_info_churn_score(series):
    return mutual_info_score(series, df_full_train.churn)

**Function-এর কাজ:**

- `series` হলো একবারে একটি categorical column, যেমন `contract` বা `gender`।
- `mutual_info_score(series, df_full_train.churn)` ওই column-এর সঙ্গে `churn`-এর MI score বের করে।
- function শুধু score return করে, যাতে `apply()` প্রতিটি column-এর জন্য একই calculation চালাতে পারে।

In [43]:
mi = df_full_train[categorical].apply(mutual_info_churn_score)
mi.sort_values(ascending=False)

contract            0.098320
onlinesecurity      0.063085
techsupport         0.061032
internetservice     0.055868
onlinebackup        0.046923
deviceprotection    0.043453
paymentmethod       0.043210
streamingtv         0.031853
streamingmovies     0.031581
paperlessbilling    0.017589
dependents          0.012346
partner             0.009968
seniorcitizen       0.009410
multiplelines       0.000857
phoneservice        0.000229
gender              0.000117
dtype: float64

### Code-টি কীভাবে কাজ করে?

1. `df_full_train[categorical]` → শুধু categorical feature-গুলোর column নেয়; target `churn`-কে feature list-এ রাখা হয়নি।
2. `.apply(mutual_info_churn_score)` → প্রতিটি column-এর ওপর function চালিয়ে একটি MI score বের করে।
3. `mi` → feature name-কে index এবং MI score-কে value হিসেবে রাখে।
4. `.sort_values(ascending=False)` → score বড় থেকে ছোট সাজায়, তাই উপরের feature-গুলো churn সম্পর্কে বেশি তথ্য দেয়।

## 5. এই dataset-এর result কীভাবে পড়বে?

| Feature | MI score (প্রায়) | Interpretation |
|---|---:|---|
| `contract` | 0.098320 | সবচেয়ে বেশি তথ্য দেয় |
| `onlinesecurity` | 0.063085 | গুরুত্বপূর্ণ signal |
| `techsupport` | 0.061032 | গুরুত্বপূর্ণ signal |
| `internetservice` | 0.055868 | গুরুত্বপূর্ণ signal |
| `paymentmethod` | 0.043210 | কিছু তথ্য দেয় |
| `partner` | 0.009968 | তুলনামূলক কম তথ্য দেয় |
| `phoneservice` | 0.000229 | প্রায় কোনো তথ্য দেয় না |
| `gender` | 0.000117 | প্রায় কোনো তথ্য দেয় না |

Score-এর মান dataset ও data distribution অনুযায়ী নির্ভর করে। এই output-এ মূলত feature-গুলোর **আপেক্ষিক ranking** দেখব; score-কে percentage হিসেবে পড়বে না।

# 3.7 — Feature Importance: Correlation



**উদ্দেশ্য:** Churn Prediction project-এ numerical feature-গুলোর সঙ্গে `churn`-এর সম্পর্ক দেখা।

**Theory reference:** Correlation-এর বিস্তারিত concept, formula, Heatmap, Multicollinearity ও Data Leakage তোমার আলাদা **Notion Correlation note**-এ আছে। এখানে শুধু project-এর প্রয়োজনীয় code ও result রাখা হয়েছে।

**আগের section থেকে লাগবে:** `df_full_train`, `numerical` এবং `churn` column।

## 1. Numerical Features

```python
numerical = ['tenure', 'monthlycharges', 'totalcharges']
```

- `tenure` = customer কত **মাস ধরে** কোম্পানির সঙ্গে আছে। যেমন `12` মানে ১২ মাস।
- `monthlycharges` = প্রতি মাসের bill; `totalcharges` = এখন পর্যন্ত মোট bill।
- `churn = 1` → customer চলে গেছে; `churn = 0` → customer থেকে গেছে।

## 2. প্রতিটি Numerical Feature-এর Correlation

`corrwith()` প্রতিটি numerical feature-এর সঙ্গে target `churn`-এর correlation বের করে।

In [44]:
df_full_train[numerical].corrwith(df_full_train.churn)

tenure           -0.351885
monthlycharges    0.196805
totalcharges     -0.196353
dtype: float64

**Expected result:**

| Feature | Correlation | কী বোঝায় |
|---|---:|---|
| `tenure` | -0.351885 | tenure বেশি হলে churn তুলনামূলক কম দেখা যায় |
| `monthlycharges` | +0.196805 | bill বেশি হলে churn তুলনামূলক বেশি দেখা যায়; সম্পর্ক দুর্বল |
| `totalcharges` | -0.196353 | দুর্বল negative relationship |

এখানে `+` বা `-` direction বোঝায়। Correlation-এর বিস্তারিত interpretation Notion note-এ আছে।

## 3. Relationship-এর Strength অনুযায়ী সাজানো

শুধু relationship-এর strength তুলনা করতে `.abs()` দিয়ে sign বাদ দিই। `sort_values(ascending=False)` বড় value আগে রাখে।

In [45]:
df_full_train[numerical].corrwith(
    df_full_train.churn
).abs().sort_values(ascending=False)

tenure            0.351885
monthlycharges    0.196805
totalcharges      0.196353
dtype: float64

**Expected ranking:**

| Feature | Absolute correlation |
|---|---:|
| `tenure` | 0.351885 |
| `monthlycharges` | 0.196805 |
| `totalcharges` | 0.196353 |

এই তিনটি feature-এর মধ্যে `tenure`-এর সঙ্গে `churn`-এর linear relationship সবচেয়ে শক্তিশালী। এটা feature-এর সম্পূর্ণ predictive importance নয়।

## 4. Group-wise Churn Rate দিয়ে Trend যাচাই

Correlation-এর result সহজে বুঝতে feature-এর value অনুযায়ী customer-দের group করে churn rate দেখি। `churn` হলো `0/1`, তাই group-এর `mean()`-ই churn rate।

In [46]:
# Tenure অনুযায়ী churn rate
print('Tenure <= 2 months:', df_full_train[df_full_train.tenure <= 2].churn.mean())
print('2 < Tenure <= 12 months:', df_full_train[(df_full_train.tenure > 2) & (df_full_train.tenure <= 12)].churn.mean())
print('Tenure > 12 months:', df_full_train[df_full_train.tenure > 12].churn.mean())

# Monthly charges অনুযায়ী churn rate
print('Monthly charges <= $20:', df_full_train[df_full_train.monthlycharges <= 20].churn.mean())
print('$20 < Monthly charges <= $50:', df_full_train[(df_full_train.monthlycharges > 20) & (df_full_train.monthlycharges <= 50)].churn.mean())
print('Monthly charges > $50:', df_full_train[df_full_train.monthlycharges > 50].churn.mean())

Tenure <= 2 months: 0.5953420669577875
2 < Tenure <= 12 months: 0.3994413407821229
Tenure > 12 months: 0.17634908339788277
Monthly charges <= $20: 0.08795411089866156
$20 < Monthly charges <= $50: 0.18340943683409436
Monthly charges > $50: 0.32499341585462205


### Result Interpretation

| Group | Churn rate (প্রায়) |
|---|---:|
| Tenure ≤ 2 months | 59.5% |
| Tenure > 2 to 12 months | 39.9% |
| Tenure > 12 months | 17.6% |
| Monthly charges ≤ $20 | 8.8% |
| Monthly charges > $20 to $50 | 18.3% |
| Monthly charges > $50 | 32.5% |

- Tenure বাড়ার সঙ্গে churn rate কমে — negative correlation-এর সঙ্গে মেলে।
- Monthly charges বাড়ার সঙ্গে churn rate বাড়ে — positive correlation-এর সঙ্গে মেলে।

এগুলো trend দেখায়; কোন feature-ই একা churn-এর সরাসরি কারণ প্রমাণ করে না।

## Quick Recap

- `corrwith()` → numerical feature-গুলোর সঙ্গে `churn`-এর correlation।
- `.abs().sort_values(ascending=False)` → absolute correlation অনুযায়ী ranking।
- Group-wise `churn.mean()` → প্রতিটি group-এর churn rate।

**Correlation-এর বাকি theory ও visualization:** আলাদা Notion Correlation note দেখো।